# Multi-agent Systems and Agentic Patterns with CrewAI

## Introduction

**CrewAI** is a powerful framework for orchestrating role-playing AI agents that work together to accomplish complex tasks. It provides a simple yet flexible API for creating multi-agent systems where each agent has specific roles, goals, and capabilities.

In this tutorial, we'll explore five fundamental agentic patterns using CrewAI with an **Azure OpenAI** deployment (`gpt-5.4-mini` or `gpt-5.6-luna`) as the model backend.

## 1. Setup & Installation

First, let's install the required libraries.

> **Note:** CrewAI supports Python 3.10–3.13 and pins many of its own dependency versions, so run this notebook in a fresh environment (for example a new Colab runtime or a dedicated virtual environment).

In [1]:
# Install required packages
# crewai: The main framework for multi-agent orchestration
# python-dotenv: Loads credentials from the .env file
# CrewAI supports Python 3.10-3.13

%pip install -q "crewai>=1.15" python-dotenv

# Windows only: pywin32 (used by the MCP SDK) normally needs a kernel restart after installation.
# Re-reading the site-packages .pth files makes it importable right away.
import site, sys, sysconfig
if sys.platform == "win32":
    for _site_dir in (sysconfig.get_paths()["purelib"], site.getusersitepackages()):
        site.addsitedir(_site_dir)

Note: you may need to restart the kernel to use updated packages.


c:\Users\Admin\Desktop\programming\accenture-bootcamp\day15\.venv-crewai\Scripts\python.exe: No module named pip


### Configure Azure OpenAI Credentials

This notebook uses an **Azure OpenAI** deployment through its OpenAI-compatible **v1 endpoint**.
Credentials are read from a `.env` file in this notebook's folder (or any parent folder). Copy `example.env` to `.env` and fill it in. On **Google Colab**, upload `.env` with the 📁 *Files* panel, or just enter the values when prompted:

| Variable | Description |
|----------|-------------|
| `AZURE_OPENAI_API_KEY` | Your Azure OpenAI API key |
| `AZURE_OPENAI_ENDPOINT` | Base URL, e.g. `https://<resource>.openai.azure.com/` (the full `.../openai/v1/` URL also works) |
| `AZURE_OPENAI_DEPLOYMENT` | Deployment (model) name: `gpt-5.4-mini` or `gpt-5.6-luna` |

If a value is missing, you'll be prompted for it.

**Security Best Practice**: Never hardcode API keys in your code. Keep them in `.env` (which must never be committed or shared).

In [2]:
import getpass
import os

from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))

if not os.environ.get("AZURE_OPENAI_API_KEY"):
    os.environ["AZURE_OPENAI_API_KEY"] = getpass.getpass("Enter your Azure OpenAI API key: ")
if not os.environ.get("AZURE_OPENAI_ENDPOINT"):
    os.environ["AZURE_OPENAI_ENDPOINT"] = input("Enter your Azure OpenAI endpoint (https://<resource>.openai.azure.com/): ")

# Accept either the resource URL or the full v1 base URL
AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"].rstrip("/").removesuffix("/openai/v1") + "/openai/v1/"
AZURE_OPENAI_DEPLOYMENT = os.environ.get("AZURE_OPENAI_DEPLOYMENT", "gpt-5.4-mini")
print(f"Using Azure OpenAI deployment: {AZURE_OPENAI_DEPLOYMENT}")

Using Azure OpenAI deployment: gpt-5.4-mini


## 2. Model Configuration

Let's create a reusable LLM configuration using CrewAI's `LLM` class, pointed at our Azure OpenAI deployment. The `openai/` prefix selects CrewAI's native OpenAI provider, and `base_url` points it at the Azure OpenAI v1 endpoint.

In [3]:
from crewai import LLM

llm = LLM(
    model=f"openai/{AZURE_OPENAI_DEPLOYMENT}",  # Prefix with openai/ so it uses the OpenAI-compatible protocol
    base_url=AZURE_OPENAI_ENDPOINT,
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
)

print(f"✅ Language model configured: {AZURE_OPENAI_DEPLOYMENT} (Azure OpenAI)")

✅ Language model configured: gpt-5.4-mini (Azure OpenAI)


## 3. Pattern A: Prompt Chaining

**Concept**: In prompt chaining, the output of one agent becomes the input for the next agent, creating a sequential pipeline of processing.

**Use Case**: When a task naturally breaks into sequential steps where each step depends on the previous one.

**Example**: Research → Summarization

> **Note:** Jupyter already runs an asyncio event loop, and CrewAI refuses to start a crew synchronously inside a running loop. In notebooks we therefore use `await crew.kickoff_async(...)`; in a plain Python script you can call `crew.kickoff(...)` instead.

In [4]:
from crewai import Agent, Task, Crew

# Agent A: Research Agent
# This agent gathers detailed information on a topic
research_agent = Agent(
    role="Research Specialist",
    goal="Gather comprehensive information about {topic}",
    backstory="You are an expert researcher with deep knowledge across multiple domains. "
              "You excel at finding relevant facts and insights.",
    llm=llm,
    verbose=True
)

# Agent B: Summary Agent
# This agent takes the research output and creates a concise summary
summary_agent = Agent(
    role="Content Summarizer",
    goal="Create a clear and concise summary of the research findings",
    backstory="You are a skilled writer who excels at distilling complex information "
              "into easily digestible summaries.",
    llm=llm,
    verbose=True
)

# Task 1: Research Task (Agent A)
# Output will be passed to the next task
research_task = Task(
    description="Research and gather detailed information about {topic}. "
                "Include key facts, recent developments, and important context.",
    agent=research_agent,
    expected_output="A comprehensive research document with key facts and insights"
)

# Task 2: Summary Task (Agent B)
# This task receives the output from research_task as context
summary_task = Task(
    description="Take the research findings and create a concise 3-paragraph summary. "
                "Focus on the most important points.",
    agent=summary_agent,
    expected_output="A clear 3-paragraph summary of the research findings",
    context=[research_task]  # This creates the chain: research_task output → summary_task input
)

# Create the crew with sequential process (default)
chain_crew = Crew(
    agents=[research_agent, summary_agent],
    tasks=[research_task, summary_task],
    verbose=True
)

# Execute the chain
print("=" * 60)
print("PATTERN A: PROMPT CHAINING EXAMPLE")
print("=" * 60)

result = await chain_crew.kickoff_async(inputs={"topic": "Quantum Computing"})
print("\n" + "=" * 60)
print("FINAL RESULT:")
print("=" * 60)
print(result)

PATTERN A: PROMPT CHAINING EXAMPLE


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 0c84490f-87ca-44d6-91dd-b1de9e6d2b07                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Research and gather detailed information about Quantum Computing. Include key facts, recent              │
│  developments, and important context.                                                                           │
│  ID: 376ace26-90ce-4526-90af-97624db5c361                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Specialist                                                                                     │
│                                                                                                                 │
│  Task: Research and gather detailed information about Quantum Computing. Include key facts, recent              │
│  developments, and important context.                                                                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Using an approximate input token count for model 'gpt-5.4-mini': assuming one token per four characters because a native token count is unavailable.


[Finalize] todos_count=0, todos_with_results=0


... (5 outputs left out)

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Summarizer                                                                                      │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Quantum computing is a fundamentally different approach to computation that uses qubits, superposition,        │
│  entanglement, and interference to process information in ways classical computers cannot. It offers the        │
│  possibility of major speedups for only certain problem types, not for all tasks. The field is still in the     │
│  NISQ era, meaning today’s devices are limited by noise, decoherence, and a lack of full error correction, but  │
│  rapid progress has been made in hardware, algorithms, software, and small-scale experimental demonstrations.   │
│                                                                                                                 │
│  The most promising long-term applications are in cryptography, chemistry, materials science, optimization,     │
│  and some specialized machine learning and simulation tasks. Shor’s algorithm could eventually break widely     │
│  used public-key systems such as RSA and ECC on sufficiently large fault-tolerant machines, while Grover’s      │
│  algorithm provides a more modest quadratic speedup for search. Quantum simulation is especially important      │
│  because quantum computers may handle molecules and materials that are too complex for classical methods,       │
│  making this area one of the strongest candidates for real-world value.                                         │
│                                                                                                                 │
│  Despite the excitement, major challenges remain: error rates are high, scaling is difficult, and large         │
│  fault-tolerant systems will require massive error-correction overhead. Multiple hardware approaches are being  │
│  pursued, including superconducting qubits, trapped ions, neutral atoms, photonics, semiconductor spins, and    │
│  topological qubits, each with distinct strengths and tradeoffs. The near-term future is likely to involve      │
│  hybrid quantum-classical workflows and continued benchmarking, while the most immediate real-world impact is   │
│  the transition to post-quantum cryptography.                                                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Take the research findings and create a concise 3-paragraph summary. Focus on the most important         │
│  points.                                                                                                        │
│  Agent: Content Summarizer                                                                                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 0c84490f-87ca-44d6-91dd-b1de9e6d2b07                                                                       │
│  Final Output: Quantum computing is a fundamentally different approach to computation that uses qubits,         │
│  superposition, entanglement, and interference to process information in ways classical computers cannot. It    │
│  offers the possibility of major speedups for only certain problem types, not for all tasks. The field is       │
│  still in the NISQ era, meaning today’s devices are limited by noise, decoherence, and a lack of full error     │
│  correction, but rapid progress has been made in hardware, algorithms, software, and small-scale experimental   │
│  demonstrations.                                                                                                │
│                                                                                                                 │
│  The most promising long-term applications are in cryptography, chemistry, materials science, optimization,     │
│  and some specialized machine learning and simulation tasks. Shor’s algorithm could eventually break widely     │
│  used public-key systems such as RSA and ECC on sufficiently large fault-tolerant machines, while Grover’s      │
│  algorithm provides a more modest quadratic speedup for search. Quantum simulation is especially important      │
│  because quantum computers may handle molecules and materials that are too complex for classical methods,       │
│  making this area one of the strongest candidates for real-world value.                                         │
│                                                                                                                 │
│  Despite the excitement, major challenges remain: error rates are high, scaling is difficult, and large         │
│  fault-tolerant systems will require massive error-correction overhead. Multiple hardware approaches are being  │
│  pursued, including superconducting qubits, trapped ions, neutral atoms, photonics, semiconductor spins, and    │
│  topological qubits, each with distinct strengths and tradeoffs. The near-term future is likely to involve      │
│  hybrid quantum-classical workflows and continued benchmarking, while the most immediate real-world impact is   │
│  the transition to post-quantum cryptography.                                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


FINAL RESULT:
Quantum computing is a fundamentally different approach to computation that uses qubits, superposition, entanglement, and interference to process information in ways classical computers cannot. It offers the possibility of major speedups for only certain problem types, not for all tasks. The field is still in the NISQ era, meaning today’s devices are limited by noise, decoherence, and a lack of full error correction, but rapid progress has been made in hardware, algorithms, software, and small-scale experimental demonstrations.

The most promising long-term applications are in cryptography, chemistry, materials science, optimization, and some specialized machine learning and simulation tasks. Shor’s algorithm could eventually break widely used public-key systems such as RSA and ECC on sufficiently large fault-tolerant machines, while Grover’s algorithm provides a more modest quadratic speedup for search. Quantum simulation is especially important because quantum computer

## 4. Pattern B: Routing

**Concept**: A router agent analyzes incoming requests and directs them to the appropriate specialized agent based on content or intent.

**Use Case**: When you have multiple specialized agents and need intelligent request distribution.

**Example**: Customer Service Router → Specialized Support Agents

In [5]:
# Specialized Agent 1: Technical Support
technical_agent = Agent(
    role="Technical Support Specialist",
    goal="Resolve technical issues and provide troubleshooting guidance",
    backstory="You are a technical expert with deep knowledge of software, hardware, "
              "and IT systems. You excel at diagnosing and solving technical problems.",
    llm=llm,
    verbose=True
)

# Specialized Agent 2: Sales Agent
sales_agent = Agent(
    role="Sales Representative",
    goal="Provide product information and help customers make purchase decisions",
    backstory="You are a knowledgeable sales professional who understands customer needs "
              "and can recommend the right products and services.",
    llm=llm,
    verbose=True
)

# Specialized Agent 3: General Information Agent
general_agent = Agent(
    role="General Information Specialist",
    goal="Answer general questions about company policies, hours, and basic information",
    backstory="You are a friendly customer service representative with comprehensive "
              "knowledge of company policies and general information.",
    llm=llm,
    verbose=True
)

# Router Agent: Analyzes queries and routes to appropriate specialist
router_agent = Agent(
    role="Customer Service Router",
    goal="Analyze customer queries and determine which specialist should handle them",
    backstory="You are an intelligent routing system that can quickly categorize customer "
              "inquiries as: TECHNICAL (tech issues), SALES (product/pricing questions), "
              "or GENERAL (policies/hours/basic info).",
    llm=llm,
    verbose=True
)

# Router Task: Determines which specialist to use
routing_task = Task(
    description="Analyze this customer query: '{query}'\n\n"
                "Determine if this is a TECHNICAL, SALES, or GENERAL inquiry. "
                "Respond with ONLY one word: TECHNICAL, SALES, or GENERAL.",
    agent=router_agent,
    expected_output="One word: TECHNICAL, SALES, or GENERAL"
)

# Create tasks for each specialist
technical_task = Task(
    description="Handle this technical inquiry: '{query}'",
    agent=technical_agent,
    expected_output="Technical support response with solution steps"
)

sales_task = Task(
    description="Handle this sales inquiry: '{query}'",
    agent=sales_agent,
    expected_output="Sales response with product recommendations"
)

general_task = Task(
    description="Handle this general inquiry: '{query}'",
    agent=general_agent,
    expected_output="General information response"
)

# Function to route queries
async def route_query(query):
    """Route a query to the appropriate specialist agent"""
    print("=" * 60)
    print(f"ROUTING QUERY: {query}")
    print("=" * 60)

    # First, determine the route
    route_crew = Crew(
        agents=[router_agent],
        tasks=[routing_task],
        verbose=True
    )

    route_result = await route_crew.kickoff_async(inputs={"query": query})
    route = str(route_result).strip().upper()

    print(f"\n→ Route determined: {route}\n")

    # Then, execute the appropriate specialist task
    if "TECHNICAL" in route:
        specialist_crew = Crew(agents=[technical_agent], tasks=[technical_task], verbose=True)
    elif "SALES" in route:
        specialist_crew = Crew(agents=[sales_agent], tasks=[sales_task], verbose=True)
    else:
        specialist_crew = Crew(agents=[general_agent], tasks=[general_task], verbose=True)

    result = await specialist_crew.kickoff_async(inputs={"query": query})

    print("\n" + "=" * 60)
    print("FINAL RESPONSE:")
    print("=" * 60)
    print(result)
    return result

# Test the routing pattern
print("\n\n" + "=" * 60)
print("PATTERN B: ROUTING EXAMPLE")
print("=" * 60 + "\n")

# Test with a technical query
await route_query("My laptop won't connect to WiFi. How do I fix this?")



PATTERN B: ROUTING EXAMPLE

ROUTING QUERY: My laptop won't connect to WiFi. How do I fix this?


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: c7237396-26e5-41b4-a6db-70b34518fd3c                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Analyze this customer query: 'My laptop won't connect to WiFi. How do I fix this?'                       │
│                                                                                                                 │
│  Determine if this is a TECHNICAL, SALES, or GENERAL inquiry. Respond with ONLY one word: TECHNICAL, SALES, or  │
│  GENERAL.                                                                                                       │
│  ID: 6125d6fb-2da2-41f0-bbe5-f3aa283960b3                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Customer Service Router                                                                                 │
│                                                                                                                 │
│  Task: Analyze this customer query: 'My laptop won't connect to WiFi. How do I fix this?'                       │
│                                                                                                                 │
│  Determine if this is a TECHNICAL, SALES, or GENERAL inquiry. Respond with ONLY one word: TECHNICAL, SALES, or  │
│  GENERAL.                                                                                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Customer Service Router                                                                                 │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  TECHNICAL                                                                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

... (8 outputs left out)

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Handle this technical inquiry: 'My laptop won't connect to WiFi. How do I fix this?'                     │
│  Agent: Technical Support Specialist                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


FINAL RESPONSE:
Here are the most effective steps to fix a laptop that won’t connect to Wi‑Fi, starting from the simplest:

## 1) Check the basics
- Make sure **Wi‑Fi is turned on** on the laptop.
- Confirm **Airplane Mode is off**.
- Verify your **router is working** by checking whether other devices can connect.
- If only your laptop is affected, the issue is likely with the laptop or its settings.

## 2) Restart everything
- **Restart your laptop**.
- **Power cycle the router/modem**:
  1. Unplug the router/modem from power.
  2. Wait 30 seconds.
  3. Plug it back in and wait until all lights stabilize.
  4. Try connecting again.

## 3) Forget and reconnect to the network
- Go to your Wi‑Fi settings.
- Select your network and choose **Forget** / **Remove**.
- Reconnect by entering the Wi‑Fi password again.

## 4) Check the password and signal
- Make sure the password is correct.
- Try moving closer to the router.
- If the network appears weak or unstable, interference may be the is

CrewOutput(raw='Here are the most effective steps to fix a laptop that won’t connect to Wi‑Fi, starting from the simplest:\n\n## 1) Check the basics\n- Make sure **Wi‑Fi is turned on** on the laptop.\n- Confirm **Airplane Mode is off**.\n- Verify your **router is working** by checking whether other devices can connect.\n- If only your laptop is affected, the issue is likely with the laptop or its settings.\n\n## 2) Restart everything\n- **Restart your laptop**.\n- **Power cycle the router/modem**:\n  1. Unplug the router/modem from power.\n  2. Wait 30 seconds.\n  3. Plug it back in and wait until all lights stabilize.\n  4. Try connecting again.\n\n## 3) Forget and reconnect to the network\n- Go to your Wi‑Fi settings.\n- Select your network and choose **Forget** / **Remove**.\n- Reconnect by entering the Wi‑Fi password again.\n\n## 4) Check the password and signal\n- Make sure the password is correct.\n- Try moving closer to the router.\n- If the network appears weak or unstable, int

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 750bcdef-5605-46ae-ab73-b9f08704ecd3                                                                       │
│  Final Output: Here are the most effective steps to fix a laptop that won’t connect to Wi‑Fi, starting from     │
│  the simplest:                                                                                                  │
│                                                                                                                 │
│  ## 1) Check the basics                                                                                         │
│  - Make sure **Wi‑Fi is turned on** on the laptop.                                                              │
│  - Confirm **Airplane Mode is off**.                                                                            │
│  - Verify your **router is working** by checking whether other devices can connect.                             │
│  - If only your laptop is affected, the issue is likely with the laptop or its settings.                        │
│                                                                                                                 │
│  ## 2) Restart everything                                                                                       │
│  - **Restart your laptop**.                                                                                     │
│  - **Power cycle the router/modem**:                                                                            │
│    1. Unplug the router/modem from power.                                                                       │
│    2. Wait 30 seconds.                                                                                          │
│    3. Plug it back in and wait until all lights stabilize.                                                      │
│    4. Try connecting again.                                                                                     │
│                                                                                                                 │
│  ## 3) Forget and reconnect to the network                                                                      │
│  - Go to your Wi‑Fi settings.                                                                                   │
│  - Select your network and choose **Forget** / **Remove**.                                                      │
│  - Reconnect by entering the Wi‑Fi password again.                                                              │
│                                                                                                                 │
│  ## 4) Check the password and signal                                                                            │
│  - Make sure the password is correct.                                                                           │
│  - Try moving closer to the router.                                                                             │
│  - If the network appears weak or unstable, interference may be the issue.                                      │
│                                                                                                                 │
│  ## 5) Disable and re-enable the Wi‑Fi adapter                                                                  │
│  ### Windows                                                                                                    │
│  - Open **Device Manager**.                           

## 5. Pattern C: Parallelization

**Concept**: Multiple agents work simultaneously on different aspects of a problem, then their results are aggregated.

**Use Case**: When you have independent sub-tasks that can be executed concurrently to save time.

**Example**: Market Research + Competitor Analysis → Unified Report

**How**: Tasks marked with `async_execution=True` run concurrently. A later task that lists them in `context` waits for all of them to finish before it starts.

In [6]:
from crewai import Process

# Agent 1: Market Research Agent (runs in parallel)
market_researcher = Agent(
    role="Market Research Analyst",
    goal="Analyze market trends, customer needs, and opportunities for {product}",
    backstory="You are a market research expert with years of experience in analyzing "
              "market dynamics, consumer behavior, and industry trends.",
    llm=llm,
    verbose=True
)

# Agent 2: Competitor Analysis Agent (runs in parallel)
competitor_analyst = Agent(
    role="Competitive Intelligence Analyst",
    goal="Analyze competitors, their strategies, and market positioning for {product}",
    backstory="You are a competitive intelligence specialist who excels at understanding "
              "competitor strengths, weaknesses, and market strategies.",
    llm=llm,
    verbose=True
)

# Agent 3: Aggregator Agent (runs after parallel tasks complete)
aggregator = Agent(
    role="Strategic Report Writer",
    goal="Combine market research and competitor analysis into a unified strategic report",
    backstory="You are a strategic analyst who excels at synthesizing multiple sources "
              "of information into actionable insights.",
    llm=llm,
    verbose=True
)

# Parallel Task 1: Market Research
market_task = Task(
    description="Conduct market research for {product}. Analyze:\n"
                "1. Market size and growth potential\n"
                "2. Target customer segments\n"
                "3. Key market trends\n"
                "4. Opportunities and challenges",
    agent=market_researcher,
    expected_output="Comprehensive market research report with key insights",
    async_execution=True,  # Runs concurrently with competitor_task
)

# Parallel Task 2: Competitor Analysis
competitor_task = Task(
    description="Analyze competitors in the {product} market. Include:\n"
                "1. Main competitors\n"
                "2. Their strengths and weaknesses\n"
                "3. Market share and positioning\n"
                "4. Competitive advantages",
    agent=competitor_analyst,
    expected_output="Detailed competitor analysis report",
    async_execution=True,  # Runs concurrently with market_task
)

# Aggregation Task: Combines results from both parallel tasks
aggregation_task = Task(
    description="Synthesize the market research and competitor analysis into a unified "
                "strategic report. Provide actionable recommendations.",
    agent=aggregator,
    expected_output="Unified strategic report with recommendations",
    context=[market_task, competitor_task]  # Waits for both tasks to complete
)

# Create the crew: both async tasks start at the same time, and the
# aggregation task waits for their results (via `context`) before it runs
parallel_crew = Crew(
    agents=[market_researcher, competitor_analyst, aggregator],
    tasks=[market_task, competitor_task, aggregation_task],
    verbose=True,
    process=Process.sequential  # Tasks 1 & 2 run concurrently; Task 3 aggregates
)

# Execute parallel pattern
print("=" * 60)
print("PATTERN C: PARALLELIZATION EXAMPLE")
print("=" * 60)
print("Note: Market research and competitor analysis run concurrently (async_execution=True)")
print("=" * 60 + "\n")

result = await parallel_crew.kickoff_async(inputs={"product": "AI-powered fitness app"})

print("\n" + "=" * 60)
print("AGGREGATED RESULT:")
print("=" * 60)
print(result)

PATTERN C: PARALLELIZATION EXAMPLE
Note: Market research and competitor analysis run concurrently (async_execution=True)



╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 8c5c4bda-7748-4cfc-b830-5dd92c488e13                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Conduct market research for AI-powered fitness app. Analyze:                                             │
│  1. Market size and growth potential                                                                            │
│  2. Target customer segments                                                                                    │
│  3. Key market trends                                                                                           │
│  4. Opportunities and challenges                                                                                │
│  ID: 4fc8efd4-b84a-4aef-973b-61047bc4562a                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Analyze competitors in the AI-powered fitness app market. Include:                                       │
│  1. Main competitors                                                                                            │
│  2. Their strengths and weaknesses                                                                              │
│  3. Market share and positioning                                                                                │
│  4. Competitive advantages                                                                                      │
│  ID: a5457960-9933-4a12-bcf7-c75fa0f93a84                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Market Research Analyst                                                                                 │
│                                                                                                                 │
│  Task: Conduct market research for AI-powered fitness app. Analyze:                                             │
│  1. Market size and growth potential                                                                            │
│  2. Target customer segments                                                                                    │
│  3. Key market trends                                                                                           │
│  4. Opportunities and challenges                                                                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Competitive Intelligence Analyst                                                                        │
│                                                                                                                 │
│  Task: Analyze competitors in the AI-powered fitness app market. Include:                                       │
│  1. Main competitors                                                                                            │
│  2. Their strengths and weaknesses                                                                              │
│  3. Market share and positioning                                                                                │
│  4. Competitive advantages                                                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

... (9 outputs left out)

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Strategic Report Writer                                                                                 │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # Unified Strategic Report: AI-Powered Fitness App Market                                                      │
│                                                                                                                 │
│  ## Executive Summary                                                                                           │
│                                                                                                                 │
│  The AI-powered fitness app market is attractive but crowded. Demand is being fueled by rising health           │
│  consciousness, widespread smartphone and wearable adoption, normalization of digital fitness, and consumer     │
│  appetite for personalization. AI strengthens the category by enabling adaptive workout plans, real-time        │
│  coaching, progress tracking, and behavior-change support.                                                      │
│                                                                                                                 │
│  At the same time, the market is retention-sensitive and increasingly competitive. The strongest players win    │
│  not just by saying they use AI, but by combining personalization with trust, ecosystem integration, content    │
│  quality, and measurable outcomes.                                                                              │
│                                                                                                                 │
│  **Strategic conclusion:** a new entrant can succeed, but only with a sharp wedge, clear target segment, and a  │
│  product that meaningfully improves adherence and results. Generic “AI fitness app” positioning is not enough.  │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## 1) Market Attractiveness Assessment                                                                         │
│                                                                                                                 │
│  ### Market Size and Growth                                                                                     │
│  The relevant market spans:                                                                                     │
│  - digital fitness apps                                                                                         │
│  - AI coaching and workout personalization                                                                      │
│  - wellness and habit-tracking tools                                                                            │
│  - connected fitness platforms                                                                                  │
│  - subscription health and fitness services                                                                     │
│                                                                                                                 │
│  This is already a multi-billion-dollar category, with the AI-enabled segment growing faster than the broader   │
│  market. Growth is being driven by:                    

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Synthesize the market research and competitor analysis into a unified strategic report. Provide          │
│  actionable recommendations.                                                                                    │
│  Agent: Strategic Report Writer                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


AGGREGATED RESULT:
# Unified Strategic Report: AI-Powered Fitness App Market

## Executive Summary

The AI-powered fitness app market is attractive but crowded. Demand is being fueled by rising health consciousness, widespread smartphone and wearable adoption, normalization of digital fitness, and consumer appetite for personalization. AI strengthens the category by enabling adaptive workout plans, real-time coaching, progress tracking, and behavior-change support.

At the same time, the market is retention-sensitive and increasingly competitive. The strongest players win not just by saying they use AI, but by combining personalization with trust, ecosystem integration, content quality, and measurable outcomes.

**Strategic conclusion:** a new entrant can succeed, but only with a sharp wedge, clear target segment, and a product that meaningfully improves adherence and results. Generic “AI fitness app” positioning is not enough.

---

## 1) Market Attractiveness Assessment

### Market 

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 8c5c4bda-7748-4cfc-b830-5dd92c488e13                                                                       │
│  Final Output: # Unified Strategic Report: AI-Powered Fitness App Market                                        │
│                                                                                                                 │
│  ## Executive Summary                                                                                           │
│                                                                                                                 │
│  The AI-powered fitness app market is attractive but crowded. Demand is being fueled by rising health           │
│  consciousness, widespread smartphone and wearable adoption, normalization of digital fitness, and consumer     │
│  appetite for personalization. AI strengthens the category by enabling adaptive workout plans, real-time        │
│  coaching, progress tracking, and behavior-change support.                                                      │
│                                                                                                                 │
│  At the same time, the market is retention-sensitive and increasingly competitive. The strongest players win    │
│  not just by saying they use AI, but by combining personalization with trust, ecosystem integration, content    │
│  quality, and measurable outcomes.                                                                              │
│                                                                                                                 │
│  **Strategic conclusion:** a new entrant can succeed, but only with a sharp wedge, clear target segment, and a  │
│  product that meaningfully improves adherence and results. Generic “AI fitness app” positioning is not enough.  │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## 1) Market Attractiveness Assessment                                                                         │
│                                                                                                                 │
│  ### Market Size and Growth                                                                                     │
│  The relevant market spans:                                                                                     │
│  - digital fitness apps                                                                                         │
│  - AI coaching and workout personalization                                                                      │
│  - wellness and habit-tracking tools                                                                            │
│  - connected fitness platforms                                                                                  │
│  - subscription health and fitness services                                                                     │
│                                                                                                                 │
│  This is already a multi-billion-dollar category, with the AI-enabled segment growing faster than the broader   │
│  market. Growth is being driven by:                   

## 6. Pattern D: Orchestrator-Worker

**Concept**: A manager/orchestrator agent breaks down complex tasks into subtasks and delegates them to worker agents sequentially.

**Use Case**: When a complex task needs to be decomposed into smaller, manageable steps that must be executed in order.

**Example**: Manager Agent → Worker Agent (Research → Outline → Draft → Edit)

In [7]:
# Manager Agent: Orchestrates the overall process
manager_agent = Agent(
    role="Project Manager",
    goal="Break down the task of writing a blog post about {topic} into clear subtasks",
    backstory="You are an experienced project manager who excels at decomposing complex "
              "tasks into manageable steps and ensuring quality execution.",
    llm=llm,
    verbose=True
)

# Worker Agent: Executes the subtasks
worker_agent = Agent(
    role="Content Worker",
    goal="Execute assigned subtasks efficiently and effectively",
    backstory="You are a versatile content professional capable of research, writing, "
              "and editing. You follow instructions precisely and deliver quality work.",
    llm=llm,
    verbose=True
)

# Step 1: Manager plans the work
planning_task = Task(
    description="Create a plan for writing a blog post about {topic}. "
                "List the key steps needed (research, outline, draft, edit).",
    agent=manager_agent,
    expected_output="A clear plan with 4 steps for creating the blog post"
)

# Step 2: Worker conducts research
research_step = Task(
    description="SUBTASK 1: Research {topic}. Gather key facts, statistics, and insights "
                "that would be valuable for a blog post.",
    agent=worker_agent,
    expected_output="Research findings with key facts and insights",
    context=[planning_task]
)

# Step 3: Worker creates outline
outline_step = Task(
    description="SUBTASK 2: Based on the research, create a detailed outline for a blog post "
                "about {topic}. Include main sections and key points.",
    agent=worker_agent,
    expected_output="Detailed blog post outline with sections and key points",
    context=[research_step]
)

# Step 4: Worker writes draft
draft_step = Task(
    description="SUBTASK 3: Using the outline, write a complete draft of the blog post. "
                "Make it engaging and informative (approximately 300 words).",
    agent=worker_agent,
    expected_output="Complete blog post draft",
    context=[outline_step]
)

# Step 5: Worker edits and finalizes
edit_step = Task(
    description="SUBTASK 4: Review and edit the draft. Improve clarity, fix any issues, "
                "and ensure the blog post is polished and ready for publication.",
    agent=worker_agent,
    expected_output="Final, polished blog post ready for publication",
    context=[draft_step]
)

# Create the orchestrator-worker crew
orchestrator_crew = Crew(
    agents=[manager_agent, worker_agent],
    tasks=[planning_task, research_step, outline_step, draft_step, edit_step],
    verbose=True,
    process=Process.sequential
)

# Execute the orchestrator-worker pattern
print("=" * 60)
print("PATTERN D: ORCHESTRATOR-WORKER EXAMPLE")
print("=" * 60)
print("Manager breaks down task → Worker executes each step")
print("=" * 60 + "\n")

result = await orchestrator_crew.kickoff_async(inputs={"topic": "The Future of Renewable Energy"})

print("\n" + "=" * 60)
print("FINAL BLOG POST:")
print("=" * 60)
print(result)

PATTERN D: ORCHESTRATOR-WORKER EXAMPLE
Manager breaks down task → Worker executes each step



╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 52586676-4ebf-4533-9544-276111f333e9                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Create a plan for writing a blog post about The Future of Renewable Energy. List the key steps needed    │
│  (research, outline, draft, edit).                                                                              │
│  ID: 5a9d48be-237f-4b30-bbf6-02e35495457f                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Project Manager                                                                                         │
│                                                                                                                 │
│  Task: Create a plan for writing a blog post about The Future of Renewable Energy. List the key steps needed    │
│  (research, outline, draft, edit).                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Project Manager                                                                                         │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Here is a clear 4-step plan for writing a blog post about **The Future of Renewable Energy**:                  │
│                                                                                                                 │
│  1. **Research the Topic**                                                                                      │
│     Gather up-to-date information on renewable energy trends, key technologies, market growth, government       │
│  policies, and challenges. Look for credible sources such as industry reports, scientific articles, and energy  │
│  news.                                                                                                          │
│                                                                                                                 │
│  2. **Create an Outline**                                                                                       │
│     Organize the main points into a logical structure. Include an introduction, major sections on current       │
│  developments and future possibilities, and a conclusion that summarizes the outlook for renewable energy.      │
│                                                                                                                 │
│  3. **Write the First Draft**                                                                                   │
│     Use the outline to draft the blog post. Focus on explaining ideas clearly, adding examples, and keeping     │
│  the tone engaging for the target audience. Don’t worry about perfection in this stage.                         │
│                                                                                                                 │
│  4. **Edit and Revise**                                                                                         │
│     Review the draft for clarity, grammar, flow, and accuracy. Strengthen weak sections, remove unnecessary     │
│  information, and make sure the final post is polished and easy to read.                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

... (19 outputs left out)

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Worker                                                                                          │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # The Future of Renewable Energy: What Comes Next for Solar, Wind, Storage, and the Clean Energy Transition    │
│                                                                                                                 │
│  Renewable energy is no longer a distant vision of the future — it’s the fastest-growing source of new          │
│  electricity around the world today. Solar and wind power are setting record capacity additions, and what was   │
│  once seen as a niche environmental solution is now a mainstream economic and infrastructure priority.          │
│                                                                                                                 │
│  So what comes next?                                                                                            │
│                                                                                                                 │
│  The future of renewable energy is not just about generating more clean power. It’s about storing it, moving    │
│  it, and managing it more intelligently. In other words, the clean energy transition is shifting from a         │
│  question of *if* to *how fast*.                                                                                │
│                                                                                                                 │
│  ## Solar and Wind Will Continue to Lead                                                                        │
│                                                                                                                 │
│  Solar power has become one of the cheapest sources of new electricity in many regions, thanks to dramatic      │
│  price declines over the past decade. It is also highly scalable, which is why it is expanding across           │
│  utility-scale solar farms, rooftops, commercial buildings, and even floating solar installations.              │
│                                                                                                                 │
│  Wind energy remains just as important. Onshore wind continues to be one of the lowest-cost options in the      │
│  right locations, while offshore wind is growing quickly as technology improves and countries look to harness   │
│  stronger coastal winds.                                                                                        │
│                                                                                                                 │
│  Together, solar and wind will likely remain the backbone of the global energy system for decades.              │
│                                                                                                                 │
│  ## Falling Costs Are Accelerating the Transition                                                               │
│                                                                                                                 │
│  The costs of solar panels, wind turbines, and batteries have dropped significantly over the last 10 to 15      │
│  years. That decline has made renewable projects competitive with — and in many cases cheaper than — fossil     │
│  fuel alternatives.                                                                                             │
│                                                        

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: SUBTASK 4: Review and edit the draft. Improve clarity, fix any issues, and ensure the blog post is       │
│  polished and ready for publication.                                                                            │
│  Agent: Content Worker                                                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


FINAL BLOG POST:
# The Future of Renewable Energy: What Comes Next for Solar, Wind, Storage, and the Clean Energy Transition

Renewable energy is no longer a distant vision of the future — it’s the fastest-growing source of new electricity around the world today. Solar and wind power are setting record capacity additions, and what was once seen as a niche environmental solution is now a mainstream economic and infrastructure priority.

So what comes next?

The future of renewable energy is not just about generating more clean power. It’s about storing it, moving it, and managing it more intelligently. In other words, the clean energy transition is shifting from a question of *if* to *how fast*.

## Solar and Wind Will Continue to Lead

Solar power has become one of the cheapest sources of new electricity in many regions, thanks to dramatic price declines over the past decade. It is also highly scalable, which is why it is expanding across utility-scale solar farms, rooftops, commercia

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 52586676-4ebf-4533-9544-276111f333e9                                                                       │
│  Final Output: # The Future of Renewable Energy: What Comes Next for Solar, Wind, Storage, and the Clean        │
│  Energy Transition                                                                                              │
│                                                                                                                 │
│  Renewable energy is no longer a distant vision of the future — it’s the fastest-growing source of new          │
│  electricity around the world today. Solar and wind power are setting record capacity additions, and what was   │
│  once seen as a niche environmental solution is now a mainstream economic and infrastructure priority.          │
│                                                                                                                 │
│  So what comes next?                                                                                            │
│                                                                                                                 │
│  The future of renewable energy is not just about generating more clean power. It’s about storing it, moving    │
│  it, and managing it more intelligently. In other words, the clean energy transition is shifting from a         │
│  question of *if* to *how fast*.                                                                                │
│                                                                                                                 │
│  ## Solar and Wind Will Continue to Lead                                                                        │
│                                                                                                                 │
│  Solar power has become one of the cheapest sources of new electricity in many regions, thanks to dramatic      │
│  price declines over the past decade. It is also highly scalable, which is why it is expanding across           │
│  utility-scale solar farms, rooftops, commercial buildings, and even floating solar installations.              │
│                                                                                                                 │
│  Wind energy remains just as important. Onshore wind continues to be one of the lowest-cost options in the      │
│  right locations, while offshore wind is growing quickly as technology improves and countries look to harness   │
│  stronger coastal winds.                                                                                        │
│                                                                                                                 │
│  Together, solar and wind will likely remain the backbone of the global energy system for decades.              │
│                                                                                                                 │
│  ## Falling Costs Are Accelerating the Transition                                                               │
│                                                                                                                 │
│  The costs of solar panels, wind turbines, and batteries have dropped significantly over the last 10 to 15      │
│  years. That decline has made renewable projects competitive with — and in many cases cheaper than — fossil     │
│  fuel alternatives.                                   

## 7. Pattern E: Evaluator-Optimizer

**Concept**: An iterative feedback loop where one agent generates content, another evaluates it, and the generator improves based on feedback.

**Use Case**: When output quality is critical and needs iterative refinement.

**Example**: Content Generator → Evaluator → Improved Generator (Feedback Loop)

In [8]:
# Generator Agent: Creates initial content
generator_agent = Agent(
    role="Content Generator",
    goal="Create compelling marketing copy for {product}",
    backstory="You are a creative copywriter who generates engaging marketing content. "
              "You're open to feedback and continuously improve your work.",
    llm=llm,
    verbose=True
)

# Evaluator Agent: Critiques the content
evaluator_agent = Agent(
    role="Marketing Copy Evaluator",
    goal="Evaluate marketing copy and provide constructive feedback",
    backstory="You are a seasoned marketing director with a keen eye for effective copy. "
              "You evaluate content based on: clarity, persuasiveness, emotional appeal, "
              "and call-to-action strength.",
    llm=llm,
    verbose=True
)

# Optimizer Agent: Improves based on feedback
optimizer_agent = Agent(
    role="Content Optimizer",
    goal="Improve marketing copy based on evaluator feedback",
    backstory="You are a skilled editor who excels at refining content based on feedback. "
              "You make targeted improvements while maintaining the original creative vision.",
    llm=llm,
    verbose=True
)

# Iteration 1: Generate initial content
generation_task = Task(
    description="Write a compelling 100-word marketing copy for {product}. "
                "Include a strong headline and a clear call-to-action.",
    agent=generator_agent,
    expected_output="Initial marketing copy with headline and call-to-action"
)

# Iteration 2: Evaluate the content
evaluation_task = Task(
    description="Evaluate the marketing copy. Rate it on:\n"
                "1. Clarity (1-10)\n"
                "2. Persuasiveness (1-10)\n"
                "3. Emotional Appeal (1-10)\n"
                "4. Call-to-Action Strength (1-10)\n\n"
                "Provide specific suggestions for improvement.",
    agent=evaluator_agent,
    expected_output="Detailed evaluation with ratings and improvement suggestions",
    context=[generation_task]
)

# Iteration 3: Optimize based on feedback
optimization_task = Task(
    description="Improve the marketing copy based on the evaluator's feedback. "
                "Address each suggestion and create a refined version.",
    agent=optimizer_agent,
    expected_output="Improved marketing copy incorporating feedback",
    context=[generation_task, evaluation_task]
)

# Iteration 4: Re-evaluate the improved version
final_evaluation_task = Task(
    description="Re-evaluate the improved marketing copy. Compare it to the original "
                "and confirm the improvements. Provide final ratings.",
    agent=evaluator_agent,
    expected_output="Final evaluation confirming improvements",
    context=[optimization_task]
)

# Create the evaluator-optimizer crew
feedback_loop_crew = Crew(
    agents=[generator_agent, evaluator_agent, optimizer_agent],
    tasks=[generation_task, evaluation_task, optimization_task, final_evaluation_task],
    verbose=True,
    process=Process.sequential
)

# Execute the feedback loop
print("=" * 60)
print("PATTERN E: EVALUATOR-OPTIMIZER EXAMPLE")
print("=" * 60)
print("Generate → Evaluate → Optimize → Re-evaluate")
print("=" * 60 + "\n")

result = await feedback_loop_crew.kickoff_async(
    inputs={"product": "An AI-powered personal finance app that helps users save money"}
)

print("\n" + "=" * 60)
print("FINAL RESULT (After Feedback Loop):")
print("=" * 60)
print(result)

PATTERN E: EVALUATOR-OPTIMIZER EXAMPLE
Generate → Evaluate → Optimize → Re-evaluate



╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 08c241e8-1d9c-44ee-b0bb-95fe101ec9dd                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Write a compelling 100-word marketing copy for An AI-powered personal finance app that helps users save  │
│  money. Include a strong headline and a clear call-to-action.                                                   │
│  ID: 20baf878-3020-4157-8349-806cbff0b7fa                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Generator                                                                                       │
│                                                                                                                 │
│  Task: Write a compelling 100-word marketing copy for An AI-powered personal finance app that helps users save  │
│  money. Include a strong headline and a clear call-to-action.                                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Generator                                                                                       │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Take Control of Your Money with AI**                                                                         │
│                                                                                                                 │
│  Meet your smarter money manager. Our AI-powered personal finance app tracks spending, spots savings            │
│  opportunities, and helps you build better habits automatically. Get personalized insights, effortless          │
│  budgeting, and real-time alerts that keep you on track—without the stress. Whether you're saving for           │
│  something big or just want more control each month, this app makes every dollar work harder for you.           │
│                                                                                                                 │
│  Stop guessing. Start saving.                                                                                   │
│                                                                                                                 │
│  **Download now and take the first step toward a healthier financial future.**                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

... (14 outputs left out)

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Marketing Copy Evaluator                                                                                │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Here’s my re-evaluation of the improved copy, compared with the original:                                      │
│                                                                                                                 │
│  ## Overall Verdict                                                                                             │
│  Yes — the improved version is noticeably stronger. It is clearer, more benefit-led, more emotionally           │
│  resonant, and the call to action is more compelling. The message now feels polished, modern, and much more     │
│  persuasive for a budgeting app audience.                                                                       │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## What Improved vs. the Original                                                                              │
│                                                                                                                 │
│  ### 1. Clarity                                                                                                 │
│  The copy explains the product quickly and in plain language:                                                   │
│  - “AI budgeting app that learns your habits”                                                                   │
│  - “automatically tracks your spending”                                                                         │
│  - “flags subscriptions and overspending”                                                                       │
│                                                                                                                 │
│  These are concrete, easy-to-understand benefits. The value proposition is immediately obvious, which is a      │
│  major improvement if the original was less direct or more feature-heavy.                                       │
│                                                                                                                 │
│  ### 2. Persuasiveness                                                                                          │
│  The copy does a better job of connecting features to outcomes:                                                 │
│  - Automatic tracking → less effort                                                                             │
│  - Alerts and suggestions → better control                                                                      │
│  - Personalized insights → smarter decisions                                                                    │
│  - “Know exactly where your money is going—and how to keep more of it”                                          │
│                                                                                                                 │
│  That last line is especially strong because it turns a feature into a meaningful financial outcome.            │
│                                                                                                                 │
│  ### 3. Emotional Appeal                               

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Re-evaluate the improved marketing copy. Compare it to the original and confirm the improvements.        │
│  Provide final ratings.                                                                                         │
│  Agent: Marketing Copy Evaluator                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


FINAL RESULT (After Feedback Loop):
Here’s my re-evaluation of the improved copy, compared with the original:

## Overall Verdict
Yes — the improved version is noticeably stronger. It is clearer, more benefit-led, more emotionally resonant, and the call to action is more compelling. The message now feels polished, modern, and much more persuasive for a budgeting app audience.

---

## What Improved vs. the Original

### 1. Clarity
The copy explains the product quickly and in plain language:
- “AI budgeting app that learns your habits”
- “automatically tracks your spending”
- “flags subscriptions and overspending”

These are concrete, easy-to-understand benefits. The value proposition is immediately obvious, which is a major improvement if the original was less direct or more feature-heavy.

### 2. Persuasiveness
The copy does a better job of connecting features to outcomes:
- Automatic tracking → less effort
- Alerts and suggestions → better control
- Personalized insights → smarter d

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 08c241e8-1d9c-44ee-b0bb-95fe101ec9dd                                                                       │
│  Final Output: Here’s my re-evaluation of the improved copy, compared with the original:                        │
│                                                                                                                 │
│  ## Overall Verdict                                                                                             │
│  Yes — the improved version is noticeably stronger. It is clearer, more benefit-led, more emotionally           │
│  resonant, and the call to action is more compelling. The message now feels polished, modern, and much more     │
│  persuasive for a budgeting app audience.                                                                       │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## What Improved vs. the Original                                                                              │
│                                                                                                                 │
│  ### 1. Clarity                                                                                                 │
│  The copy explains the product quickly and in plain language:                                                   │
│  - “AI budgeting app that learns your habits”                                                                   │
│  - “automatically tracks your spending”                                                                         │
│  - “flags subscriptions and overspending”                                                                       │
│                                                                                                                 │
│  These are concrete, easy-to-understand benefits. The value proposition is immediately obvious, which is a      │
│  major improvement if the original was less direct or more feature-heavy.                                       │
│                                                                                                                 │
│  ### 2. Persuasiveness                                                                                          │
│  The copy does a better job of connecting features to outcomes:                                                 │
│  - Automatic tracking → less effort                                                                             │
│  - Alerts and suggestions → better control                                                                      │
│  - Personalized insights → smarter decisions                                                                    │
│  - “Know exactly where your money is going—and how to keep more of it”                                          │
│                                                                                                                 │
│  That last line is especially strong because it turns a feature into a meaningful financial outcome.            │
│                                                                                                                 │
│  ### 3. Emotional Appeal                              

## 8. Conclusion: When to Use Each Pattern

Now that we've explored the five fundamental agentic patterns, let's summarize when to use each one:

### Pattern Selection Guide

| Pattern | Best Used When | Key Benefits | Example Use Cases |
|---------|---------------|--------------|-------------------|
| **Prompt Chaining** | Tasks naturally flow in sequence, each step depends on the previous | Simple, predictable, easy to debug | Data pipeline (extract → transform → load), Research → Summary |
| **Routing** | You have multiple specialized agents and need dynamic selection | Efficient resource use, specialized expertise | Customer service routing, Intent classification, Dynamic tool selection |
| **Parallelization** | Independent subtasks can run simultaneously | Time efficiency, concurrent processing | Market research + competitor analysis, Multiple data source aggregation |
| **Orchestrator-Worker** | Complex tasks need decomposition into managed steps | Clear structure, scalable delegation | Project planning, Multi-step content creation, Complex workflows |
| **Evaluator-Optimizer** | Quality is critical and iterative improvement needed | High-quality output, continuous refinement | Content creation, Code review, Strategic planning |

### Combining Patterns

These patterns aren't mutually exclusive! In production systems, you might combine them:
- **Routing + Parallelization**: Route queries to multiple specialists who work in parallel
- **Orchestrator-Worker + Evaluator-Optimizer**: Orchestrate a process with quality checks at each step
- **Prompt Chaining + Routing**: Chain multiple steps where each step routes to different specialists

### Key Takeaways

1. **Start Simple**: Begin with Prompt Chaining for linear workflows
2. **Add Intelligence**: Use Routing when you need dynamic decision-making
3. **Scale Performance**: Apply Parallelization for time-critical operations
4. **Manage Complexity**: Use Orchestrator-Worker for complex, multi-step tasks
5. **Ensure Quality**: Implement Evaluator-Optimizer when output quality is paramount

### Next Steps

- Experiment with each pattern using your own use cases
- Try combining patterns for more sophisticated workflows
- Explore CrewAI's additional features: custom tools, memory, and advanced configurations
- Monitor agent performance and iterate on your designs

Happy building with CrewAI! 🚀